# Session 1: a table is a grid, a query asks it a question
**A. Real-world problem.** A school keeps marks for hundreds of students. The teacher asks
"who scored at least 60?" or "who is in grade 9 or above and scored at least 70?". Nobody
wants to read every row. A *database* keeps the rows in a *table* and a *query* asks the
question for us. All names and marks here are synthetic.

**Plan.** (1) build a table of 12 rows, (2) ask a question in plain Python, (3) ask the same
question in SQL with Python's built-in `sqlite3` (nothing to install), (4) check that both
answers are identical, (5) plot how the answer changes with the cut-off.

In [ ]:
import pathlib
import sqlite3

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

COLUMNS = ("id", "name", "grade", "marks")
ROWS = [
    (1, "Asha", 8, 72), (2, "Bilal", 8, 45), (3, "Chitra", 9, 88), (4, "Dev", 9, 61),
    (5, "Esha", 8, 90), (6, "Farid", 10, 55), (7, "Gita", 10, 67), (8, "Hari", 9, 39),
    (9, "Indu", 10, 95), (10, "Jai", 8, 58), (11, "Kavya", 9, 76), (12, "Lalit", 10, 49),
]

## Step 1: the table
A table is a grid. Each **row** is one student, each **column** is one fact about every
student. In SQL we create it once and then put the rows in. `:memory:` means the database
lives only in RAM and disappears when the program ends, which is perfect for learning.

In [ ]:
def make_table():
    conn = sqlite3.connect(":memory:")
    conn.execute("CREATE TABLE students (id INTEGER PRIMARY KEY, name TEXT, grade INTEGER, marks INTEGER)")
    conn.executemany("INSERT INTO students VALUES (?, ?, ?, ?)", ROWS)
    conn.commit()
    return conn

## Step 2: a question in plain Python
"Keep the rows whose marks are at least `min_marks` and whose grade is at least `min_grade`."
A **filter** keeps the rows where the condition is true and drops the others.

In [ ]:
def python_filter(rows, min_marks, min_grade=0):
    return [r for r in rows if r[3] >= min_marks and r[2] >= min_grade]

## Step 3: the same question in SQL

    SELECT id, name, grade, marks   -- which columns to show
    FROM students                   -- which table
    WHERE marks >= 60 AND grade >= 8   -- the filter
    ORDER BY id                     -- the order of the answer

The `?` marks are *parameters*: Python fills them in safely (Session 3 shows why that matters).

In [ ]:
def sql_filter(conn, min_marks, min_grade=0):
    sql = "SELECT id, name, grade, marks FROM students WHERE marks >= ? AND grade >= ? ORDER BY id"
    return conn.execute(sql, (min_marks, min_grade)).fetchall()


def numpy_count(min_marks, min_grade=0):
    """A third, independent way: boolean masks on numpy columns. Returns how many rows pass."""
    grade = np.array([r[2] for r in ROWS])
    marks = np.array([r[3] for r in ROWS])
    return int(np.sum((marks >= min_marks) & (grade >= min_grade)))


def count_curve(thresholds, min_grade=0):
    """How many rows pass for each cut-off (the step curve of the second figure)."""
    return [len(python_filter(ROWS, t, min_grade)) for t in thresholds]

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    conn = make_table()
    kept = sql_filter(conn, 60, 8)
    print("SQL answer, marks >= 60:", [r[1] for r in kept], "->", len(kept), "of", len(ROWS))
    print("Python answer matches SQL:", kept == python_filter(ROWS, 60, 8))
    print("numpy count:", numpy_count(60, 8))
    worked = sql_filter(conn, 70, 9)
    print("grade >= 9 and marks >= 70:", [r[1] for r in worked])

    # figure 1: the table with a filter
    fig, ax = plt.subplots(figsize=(6.4, 4.8))
    ax.set_xlim(0, 4)
    ax.set_ylim(len(ROWS) + 1, 0)
    ax.axis("off")
    keep_ids = {r[0] for r in kept}
    for j, name in enumerate(COLUMNS):
        ax.text(j + 0.5, 0.5, name, ha="center", va="center", weight="bold")
    for i, row in enumerate(ROWS, start=1):
        hit = row[0] in keep_ids
        ax.add_patch(plt.Rectangle((0, i), 4, 1, color="#2a9d8f" if hit else "#e5e5e5", alpha=0.55 if hit else 0.35))
        for j, v in enumerate(row):
            ax.text(j + 0.5, i + 0.5, str(v), ha="center", va="center", color="black" if hit else "#777")
    ax.set_title(f"WHERE marks >= 60 AND grade >= 8: {len(kept)} of {len(ROWS)} rows kept (green)")
    fig.tight_layout()
    fig.savefig(OUT / "table_filter.png", dpi=120)

    # figure 2: number of rows kept against the cut-off
    ts = np.arange(0, 101)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.step(ts, count_curve(ts, 0), where="post", label="all grades")
    ax2.step(ts, count_curve(ts, 9), where="post", label="grade 9 and above")
    ax2.set_xlabel("cut-off marks (marks)")
    ax2.set_ylabel("rows kept (rows)")
    ax2.set_title("Raising the cut-off keeps fewer rows")
    ax2.grid(alpha=0.3)
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "rows_vs_cutoff.png", dpi=120)

## Where this stops
One table and one filter is the whole of Session 1. Session 2 adds counting, averaging,
grouping and the question of how to split data into several tables.

## Coding tasks
1. **Easy.** Change the cut-off in `lesson.py` to 50 and 90. Predict how many rows are kept each time, then run it and compare.
2. **Stretch.** Write a SQL query that lists the names of the students in grade 8, ordered by marks from high to low, and check it against a sorted Python list of the same rows.
3. **Explain.** Explain in your own words the difference between a row and a column using the student table, and say which of them the WHERE condition removes.

## Assessment
- **A. Conceptual.** Does running a SELECT query change the table? Why does the order of rows in an answer need an ORDER BY?
- **B. Numerical.** How many rows have grade 10 and marks of at least 50? What percentage of the whole table is that?
- **C. Graph and simulation interpretation.** On the staircase graph for all grades, how many rows are kept at a cut-off of 90, and what is the first whole-number cut-off that keeps 0 rows?
- **D. Design and optimisation challenge.** Design: a scholarship interview should include exactly the top 3 students out of 12 (25 percent) by marks. Choose a cut-off for 'marks >= cut-off'. Which cut-offs work, and which one would you pick?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    conn = L.make_table()
    n = conn.execute("SELECT COUNT(*) FROM students").fetchone()[0]
    checks.append(("the table has 12 rows and 4 columns", n == 12 and len(L.COLUMNS) == 4))
    kept = L.sql_filter(conn, 60, 8)
    checks.append(("marks >= 60 keeps 7 of 12 rows", len(kept) == 7))
    checks.append(("SQL and plain Python give identical rows", kept == L.python_filter(L.ROWS, 60, 8)))
    checks.append(("numpy mask counts the same 7 rows", L.numpy_count(60, 8) == 7))
    names = [r[1] for r in L.sql_filter(conn, 70, 9)]
    checks.append(("worked example: grade >= 9 and marks >= 70 keeps Chitra, Indu, Kavya (3 rows)",
                   names == ["Chitra", "Indu", "Kavya"] and len([r for r in L.ROWS if r[2] >= 9]) == 8))
    checks.append(("a cut-off of 0 keeps every row", len(L.sql_filter(conn, 0)) == 12))
    checks.append(("a cut-off of 101 keeps no rows", L.sql_filter(conn, 101) == []))
    boundary = [r[3] for r in L.sql_filter(conn, 61)]
    checks.append(("the boundary is inclusive: marks 61 is kept by >= 61", 61 in boundary))
    curve = L.count_curve(range(0, 101))
    checks.append(("raising the cut-off never increases the count", all(a >= b for a, b in zip(curve, curve[1:]))))
    checks.append(("grade >= 9 and marks >= 75 keeps exactly 3 rows (the app challenge)", len(L.sql_filter(conn, 75, 9)) == 3))
    checks.append(("grade >= 10 keeps 4 rows", len(L.sql_filter(conn, 0, 10)) == 4))
    checks.append(("grade 10 with marks >= 50 keeps 3 rows, 25 percent of the table", len(L.sql_filter(conn, 50, 10)) == 3 and 100 * 3 / 12 == 25))
    allc = L.count_curve(range(0, 101))
    checks.append(("staircase: 2 rows at cut-off 90, 1 at 95, 0 at 96", allc[90] == 2 and allc[95] == 1 and allc[96] == 0))
    checks.append(("design: cut-offs 77 to 88 keep exactly 3 rows, 76 keeps 4",
                   all(len(L.sql_filter(conn, t)) == 3 for t in range(77, 89)) and len(L.sql_filter(conn, 76)) == 4))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
